In [1]:
import dask
import time
import toolviper
import random
import webbrowser
import pathlib

# ====== Testing ====== #

import asyncio

# ===================== #


import toolviper.utils.logger as logger

import toolviper.dask.client as client

from xradio.measurement_set import convert_msv2_to_processing_set
from xradio.measurement_set import open_processing_set

In [2]:
# def pipeline_a(x):
#     import time
#     time.sleep(10)  # Simulating heavy work
#     return x * 2

# def pipeline_b(x):
#     import time
#     time.sleep(10)
#     return x + 100

# async def manage_jobs():
#     async with client.local_client(
#         cores=12,
#         asynchronous=True,
#         log_params={
#             "log_to_file":False,
#             "log_to_term":True,
#             "log_level":"DEBUG" 
#         },
#         worker_log_params={
#             "log_to_file":False,
#             "log_to_term":True,
#             "log_level":"INFO" 
#         }
#     )as c:
#         logger.info(c.dashboard_link) 
#         logger.info("Submitting both graphs to the cluster...")
        
#         # 2. Submit graph 1 (runs in the background)
#         future_a = c.submit(pipeline_a, 10)
        
#         # 3. Submit graph 2 (runs in the background at the same time)
#         future_b = c.submit(pipeline_b, 10)
        
#         # 4. Use asyncio.gather to await both concurrently
#         # client.gather is a coroutine when client is asynchronous
#         results = await asyncio.gather(
#             c.gather(future_a),
#             c.gather(future_b)
#         )

# async def submit():
#     await manage_jobs()

In [ ]:
#result = await submit()

In [3]:
client = client.local_client(
    cores=12,
    asynchronous=False,
    log_params={
        "log_to_file":False,
        "log_to_term":True,
        "log_level":"DEBUG" 
    },
    worker_log_params={
        "log_to_file":False,
        "log_to_term":True,
        "log_level":"INFO" 
    }
)

[2026-07-16 09:50:50,378]  WARNING      client:  It is recommended that the local cache directory be set using the dask_local_dir parameter. 


/opt/homebrew/Caskroom/mambaforge/base/envs/graphviper-i62/lib/python3.13/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 59102 instead
  warnings.warn(


[2026-07-16 09:50:51,471]    DEBUG      client:  Loading plugin module: <class 'worker.DaskWorker'>
[2026-07-16 09:50:53,943]     INFO      client:  Client <MenrvaClient: 'tcp://127.0.0.1:59103' processes=12 threads=12, memory=8.74 GiB> 


In [ ]:
client.dashboard_link

#### Utilities

In [ ]:
def generate_delay(n=1, m=2):
    time.sleep(random.uniform(n, m))

def args_checker(args):
    if "delay" in args[0].keys():
        if args[0]['task_coords']['antenna_name']['data'] == args[0]['delay']:
            return True

    return False

def write_image(file_name, spw, antenna, field):

    path = pathlib.Path(file_name)
    logger.info(str(path))
    if not path.exists():
        logger.error(f"File: {str(path)} not found.")

    data_path = path.joinpath(spw).joinpath(field).joinpath(antenna)
    logger.info(str(data_path))
    data_path.mkdir(exist_ok=True, parents=True)
    
    with open(str(data_path.joinpath(f"some_data_{spw}.{field}.{antenna}.image.ps.zarr")), "w") as file:
        file.write(f"{spw}\t{field}\t{antenna}")

        

### Make Test Functions 

In [ ]:
UPPER_DELAY_LIMIT=2

# Distributed #[field, spw, antenna][EB]
def deviation_mask_heuristic(*args, **kwargs):
    logger.info("Deviation mask heuristic ... [field, spw, antenna]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Distributed #[field,spw][EB]
def line_analysis(*args, **kwargs):
    logger.info("Line analysis... [field, spw]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Distributed #[field,spw, antenna, pol][EB]
def fitting_heuristic(*args, **kwargs):
    logger.info("Fitting Heuristic ... [field, spw, antenna, pol]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Distributed #[field,spw, antenna, pol][EB]
def baseline_subtraction(*args, **kwargs):
    logger.info("Baseline subtraction ... [field, spw, antenna, pol]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Distributed #[field,spw, antenna, pol][EB]
def heuristic_plots_per_eb(*args, **kwargs):
    logger.info("Pre-EB heuristic plotting ... [field, spw, antenna, pol]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Distributed #[field, spw]
def heuristic_plots(*args, **kwargs):
    logger.info("Heuristic plotting ... [field, spw]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

### Convert the MSv2 to a Processing Set and Open

In [ ]:
pathlib.Path().cwd()

In [ ]:
# file_path = pathlib.Path().cwd().joinpath("data/uid___A002_Xced5df_Xf9d9.small.ms")
# processing_set = pathlib.Path().cwd().joinpath("data/uid___A002_Xced5df_Xf9d9.small.ms.ps.zarr")

# #if processing_set.exists():
# convert_msv2_to_processing_set(
#    in_file=str(file_path),
#    out_file=str(processing_set),
#    persistence_mode="w",
#    partition_scheme = ["FIELD_ID", "ANTENNA1"]
#    #partition_scheme = ["FIELD_ID"]
# )

In [4]:
processing_set = str("/Users/joshua/Development/processing_set/uid___A002_Xcd07af_X70c7.2spws.80000chans.lsrk.ps.zarr")

ps = open_processing_set(
    ps_store=str(processing_set),
    scan_intents=["OBSERVE_TARGET#ON_SOURCE"]
)

In [ ]:
ps["uid___A002_Xcd07af_X70c7.2spws.80000chans.lsrk_0"].antenna_xds.ds

### Demonstrate the Simple Node Building Tool

In [5]:
graph = toolviper.utils.sd.graph.Graph.from_dataset(ps)

[2026-07-16 09:51:05,396]     INFO      client:  Creating graph from dataset ... 


In [6]:
# graph.filter(leaves=["uid___A002_Xc1469f_X1c33_65", "uid___A002_Xc1469f_X1c33_66"])

In [7]:
graph.datatree

<xarray.DataTree>
Group: /
│   Attributes:
│       type:     processing_set
├── Group: /uid___A002_Xcd07af_X70c7.2spws.80000chans.lsrk_0
│   │   Dimensions:                     (time: 2730, antenna_name: 1, frequency: 80000,
│   │                                    polarization: 2)
│   │   Coordinates:
│   │     * time                        (time) float64 22kB 1.526e+09 ... 1.526e+09
│   │       field_name                  (time) <U31 339kB dask.array<chunksize=(1365,), meta=np.ndarray>
│   │       scan_name                   (time) <U21 229kB dask.array<chunksize=(2730,), meta=np.ndarray>
│   │     * antenna_name                (antenna_name) <U9 36B 'PM01_T703'
│   │       telescope_name              (antenna_name) <U4 16B dask.array<chunksize=(1,), meta=np.ndarray>
│   │     * frequency                   (frequency) float64 640kB 2.302e+11 ... 2.312...
│   │     * polarization                (polarization) <U2 16B 'XX' 'YY'
│   │   Data variables:
│   │       EFFECTIVE_INTEGRATION_TIME  (time, antenna_name) float64 22kB dask.array<chunksize=(2730, 1), meta=np.ndarray>
│   │       FLAG                        (time, antenna_name, frequency, polarization) bool 437MB dask.array<chunksize=(2730, 1, 80000, 2), meta=np.ndarray>
│   │       SPECTRUM                    (time, antenna_name, frequency, polarization) float32 2GB dask.array<chunksize=(2730, 1, 80000, 2), meta=np.ndarray>
│   │       TIME_CENTROID               (time, antenna_name) float64 22kB dask.array<chunksize=(2730, 1), meta=np.ndarray>
│   │       WEIGHT                      (time, antenna_name, frequency, polarization) float32 2GB dask.array<chunksize=(2730, 1, 80000, 2), meta=np.ndarray>
│   │   Attributes:
│   │       schema_version:    4.0.0
│   │       creator:           {'software_name': 'xradio', 'version': '1.2.0'}
│   │       creation_date:     2026-06-01T08:16:15.867847+00:00
│   │       type:              spectrum
│   │       data_groups:       {'base': {'correlated_data': 'SPECTRUM', 'flag': 'FLAG...
│   │       observation_info:  {'observer': ['srams'], 'release_date': '1858-11-17T00...
│   │       processor_info:    {'type': 'CORRELATOR', 'sub_type': 'ALMA_ACA'}
│   ├── Group: /uid___A002_Xcd07af_X70c7.2spws.80000chans.lsrk_0/antenna_xds
│   │       Dimensions:                 (antenna_name: 1, cartesian_pos_label: 3,
│   │                                    receptor_label: 2)
│   │       Coordinates:
│   │           mount                   (antenna_name) <U6 24B dask.array<chunksize=(1,), meta=np.ndarray>
│   │           station_name            (antenna_name) <U4 16B dask.array<chunksize=(1,), meta=np.ndarray>
│   │           telescope_name          (antenna_name) <U4 16B dask.array<chunksize=(1,), meta=np.ndarray>
│   │         * cartesian_pos_label     (cartesian_pos_label) <U1 12B 'x' 'y' 'z'
│   │         * receptor_label          (receptor_label) <U5 40B 'pol_0' 'pol_1'
│   │           polarization_type       (antenna_name, receptor_label) <U1 8B dask.array<chunksize=(1, 2), meta=np.ndarray>
│   │       Data variables:
│   │           ANTENNA_DISH_DIAMETER   (antenna_name) float64 8B dask.array<chunksize=(1,), meta=np.ndarray>
│   │           ANTENNA_POSITION        (antenna_name, cartesian_pos_label) float64 24B dask.array<chunksize=(1, 3), meta=np.ndarray>
│   │           ANTENNA_RECEPTOR_ANGLE  (antenna_name, receptor_label) float64 16B dask.array<chunksize=(1, 2), meta=np.ndarray>
│   │       Attributes:
│   │           type:                    antenna
│   │           relocatable_antennas:    True
│   │           overall_telescope_name:  ALMA
│   ├── Group: /uid___A002_Xcd07af_X70c7.2spws.80000chans.lsrk_0/field_and_source_base_xds
│   │       Dimensions:                           (field_name: 1, sky_dir_label: 2,
│   │                                              line_label: 1)
│   │       Coordinates:
│   │         * field_name                        (field_name) <U31 124B 'AFGL_3068_1'
│   │           source_name                       (

In [8]:
graph.make_coordinates(coords=["field_name", "antenna_name"])

[2026-07-16 09:51:07,623]     INFO      client:  Making field coordinate ... 
[2026-07-16 09:51:07,656]     INFO      client:  Making antenna coordinate ... 


In [9]:
graph.coordinates

In [10]:
graph.build_node(ps_partition = ["field_name", "antenna_name"])

[2026-07-16 09:51:09,560]    DEBUG      client:  chunk_index: 0, (np.int64(0), np.int64(0))
[2026-07-16 09:51:09,560]    DEBUG      client:  chunk_index: 1, (np.int64(0), np.int64(1))
[2026-07-16 09:51:09,561]    DEBUG      client:  chunk_index: 2, (np.int64(0), np.int64(2))
[2026-07-16 09:51:09,561]    DEBUG      client:  chunk_index: 3, (np.int64(1), np.int64(0))
[2026-07-16 09:51:09,561]    DEBUG      client:  chunk_index: 4, (np.int64(1), np.int64(1))
[2026-07-16 09:51:09,561]    DEBUG      client:  chunk_index: 5, (np.int64(1), np.int64(2))
[2026-07-16 09:51:09,562]    DEBUG      client:  chunk_index: 6, (np.int64(2), np.int64(0))
[2026-07-16 09:51:09,562]    DEBUG      client:  chunk_index: 7, (np.int64(2), np.int64(1))
[2026-07-16 09:51:09,562]    DEBUG      client:  chunk_index: 8, (np.int64(2), np.int64(2))
[2026-07-16 09:51:09,562]    DEBUG      client:  chunk_index: 9, (np.int64(3), np.int64(0))
[2026-07-16 09:51:09,562]    DEBUG      client:  chunk_index: 10, (np.int64(3), 

In [ ]:
graph.node_mapping

In [ ]:
graph.map(function=line_analysis, parameters={"parameter": False}, connect=None)             # Adding make_workflow=True here will allow you to produce results without a gather.
graph.reduce(parameters={"parameter": False}, function=fitting_heuristic, mode="tree")

In [ ]:
graph.node_mapping

In [ ]:
dask.visualize(graph._graph, filename="map_graph")

In [ ]:
graph.reset()

### Example of `hsd_baseline()` using GraphViper Framework

In [ ]:
# Spawn dashboard window in a separate tab,
# comment out if you don't want this to spawn.
webbrowser.open(url=client.dashboard_link)

def hsd_imaging(processing_set, leaves=None, file_name="no_name_test.zarr"):
    # Serial processing
    input_imaging_parameters(file_name=file_name)
    set_group()

    # Begin Preprocessing
    ps = open_processing_set(
        ps_store=str(processing_set),
        scan_intents=["OBSERVE_TARGET#ON_SOURCE"]
    )

    job = toolviper.utils.sd.graph.Graph.from_dataset(ps)

    # If you want to filter sub-dataset
    if leaves:
        job.filter(leaves=leaves)

    # We are actually doing [field, spw, antenna] here, but the processing set is already split up by spw.
    job.make_coordinates(coords=["antenna_name", "field_name"])
    job.build_node(ps_partition = ["spectral_window_name", "field_name"])
    
    # Start Dask processing - building job graph
    job.map(
        function=per_imaging_generation, 
        parameters={
        "delay": "PM03_T701",
        "file_name": file_name
        },
        connect=None, 
        make_workflow=False
    )
    
    job.reduce(
        parameters={"delay": 'PM03_T701'}, 
        function=combine_per_antenna_stokes, 
        mode="single_node"
    )

    # Finish Dask processing - compute job graph
    job.compute()
    
    make_weblog_entry()
    

In [ ]:
hsd_imaging(
    file_name="data.ouput.sd.du",
    processing_set=processing_set, 
    leaves=["uid___A002_Xc1469f_X1c33_65"]
)

### Example: Linked Graphs

In [ ]:
def first_function():
    pass

def second_function():
    pass

def gather():
    pass

def connected_graph(
    file_name="connected.ouput.sd.du",
    processing_set=processing_set, 
    leaves=["uid___A002_Xc1469f_X1c33_65"]
):
    from xradio.measurement_set import open_processing_set
    
    ps = open_processing_set(
        ps_store=str(processing_set),
        scan_intents=["OBSERVE_TARGET#ON_SOURCE"]
    )

    job = toolviper.utils.sd.graph.Graph.from_dataset(ps)

    # If you want to filter sub-dataset
    if leaves:
        job.filter(leaves=leaves)

    # We are actually doing [field, spw, antenna] here but the processing set is already split up by spw.
    job.make_coordinates(coords=["antenna_name", "field_name"])
    job.build_node(ps_partition = ["spectral_window_name", "field_name"])
    
    # Start Dask processing - building job graph
    job.map(
        function=first_function, 
        parameters={
            "file_name": file_name
        },
        connect=None, 
        make_workflow=False
    )

    job.make_coordinates(coords=["antenna_name", "field_name"])
    job.build_node(ps_partition = ["spectral_window_name", "field_name", "antenna_name"])
    
    job.map(
        function=second_function, 
        parameters={
            "file_name": file_name
        },
        connect=first_function, 
        make_workflow=False
    )
    
    job.reduce(
        parameters={"delay": 'PM03_T701'}, 
        function=gather, 
        mode="single_node"
    )

    return job


In [ ]:
j = connected_graph()

In [ ]:
j.visualize()

In [ ]:
client.close()